# Network Analysis with External APIs


In the previous section, we performed network analysis using a locally built graph and the `networkx` library.

An alternative approach is to use **external routing API services**: the computations are handled server-side, and we retrieve the results on demand.

To work with these services, it helps to understand how APIs are structured: how requests are formed, what parameters they accept, and how to handle the responses.

We begin with the basic principles of working with APIs, then walk through three routing services: **OSRM**, **OpenRouteService**, and **GraphHopper**.


## 0. Importing Libraries


In [ ]:
import requests
import polyline
import geopandas as gpd

from shapely.geometry import LineString

- [**Requests**](https://docs.python-requests.org/) (`requests`) – a Python library for sending HTTP requests and working with web services (APIs). You use it to retrieve data from servers (GET), submit data (POST), and handle responses, JSON included. It simplifies interaction with external APIs and is widely used for fetching and exchanging data over the internet.

- [**polyline**](https://pypi.org/project/polyline/) (`polyline`) – a small library for decoding the encoded polyline format that many routing services use to compress route geometry. We use it below to turn an API response into a list of coordinates.

## 1. Application Programming Interface (API)


An **API (Application Programming Interface)** is how one program talks to another.

For our purposes it is a web service: we send it a request with some parameters – a pair of coordinates, say – and it sends back data, such as a route or a distance.

That exchange goes over an **HTTP request**, the standard way a client (our program) and a server pass data between them.


### 1.1. How an API Request is Structured

An API request usually has three parts:

- a **URL** – the address of the service (e.g. `https://api.service.com/route`);
- **parameters** – the input data (coordinates, transport mode, etc.);
- **headers** – additional metadata (e.g. an API key or the expected response format).


### 1.2. Request Methods

HTTP has several request methods, each for a different kind of operation:

- **GET** – retrieves data from the server. Parameters are passed in the URL.

- **POST** – sends data to the server. Parameters are passed in the request body. Commonly used for computations.

- **PUT** – fully replaces an existing resource on the server.

- **PATCH** – partially updates a resource (modifies specific fields only).

- **DELETE** – removes a resource from the server.

We will mostly use **GET** and **POST**. Which one an operation needs is always in the service's documentation, so read it carefully.


### 1.3. Reading the Documentation

Every API has documentation that describes:

- the available endpoints (e.g. routing, matrix, isochrones);
- the required parameters;
- the request method to use (GET, POST, etc.);
- the format of the response.

Many services require an **API key**, a string that identifies you to the service. It is how they control access, enforce rate limits and keep track of usage.

### 1.4. The `requests` Library

We send our HTTP requests from Python with the `requests` library.

The server answers with a response that carries:

- a **status code** – e.g. `200` for success, `400` for a bad request, `401` for an authorisation error;
- a **response body** – the data, most commonly in JSON format;
- **response headers** – additional information about the result.

### 1.5. Checking the Response Status

After sending a request, confirm that the server processed it successfully.

The `.status_code` attribute holds the **HTTP status code**.

The most common codes are:

- `200` – the request succeeded; proceed to handling the response;
- `4xx` – a client-side error (e.g. invalid URL, bad parameters, or an incorrect API key);
- `5xx` – a server-side error.

If the code is **not `200`**, do not parse the result – read the response body instead. Between the code and the body, the reason is usually obvious.

_Note: many routing services offer ready-made Python client libraries, but in this section we work directly with the API to develop a solid understanding of how requests work._


## 2. APIs for Network Analysis


In this section, we will work with several popular APIs and use them to solve common network analysis tasks:

- finding shortest routes;
- retrieving distances and travel times;
- generating isochrones;
- computing distance matrices.

Each service has its own request format, set of features, and rate limits – all described in its documentation. For each service, we will work through one or more examples, walking through the full workflow from forming a request to processing the response and visualising the result.


### 2.1. Open Source Routing Machine (OSRM)

**OSRM (Open Source Routing Machine)** is a fast, open-source routing engine built on OpenStreetMap data.

OSRM does not require an API key – requests can be sent directly.

[Documentation](https://project-osrm.org/docs/v5.24.0/api/)


#### 2.1.1. The Task

The OSRM documentation describes how to request a route between two points.

The endpoint is:

```
/route/v1/{profile}/{coordinates}
```

where:

- `profile` – the transport mode (e.g. `driving`);
- `coordinates` – a list of points in `longitude,latitude` format, separated by `;`.

Additional query parameters can also be passed, for example `overview=full` to include the full route geometry in the response.


#### 2.1.2. Building the Request URL

Build the full URL for the API call.
We define the start and end coordinates and format them as the service expects.


In [ ]:
base_url_osrm = "https://router.project-osrm.org/route/v1/driving/"

# Coordinates (longitude, latitude)
start_coords_osrm = [16.3725, 48.2082]   # Stephansplatz
end_coords_osrm = [16.3122, 48.1847]     # Schönbrunn Palace

# Format coordinates as strings
start_osrm = f"{start_coords_osrm[0]},{start_coords_osrm[1]}"
end_osrm = f"{end_coords_osrm[0]},{end_coords_osrm[1]}"

# Build the full URL
url_osrm = f"{base_url_osrm}{start_osrm};{end_osrm}?overview=full"

> **Important:** different APIs use different coordinate ordering.
> Some services expect longitude, latitude; others expect latitude, longitude. Always check the documentation before sending a request.


#### 2.1.3. Sending the Request


Send the HTTP request with the `requests` library.


In [ ]:
response_osrm = requests.get(url_osrm)

#### 2.1.4. Checking the Response Status


Before processing the response, let's confirm that the request succeeded.


In [ ]:
response_osrm.status_code

If the status is `200`, we can proceed to parse the result. Any other code indicates an error – inspect the response body to understand what went wrong.


#### 2.1.5. Parsing the Response

The server returns a JSON response, which can be converted to a Python dictionary using the `.json()` method.


In [ ]:
data_osrm = response_osrm.json()

The top-level keys show how the response is organised. A full description of the response schema is available in the API documentation.


In [ ]:
data_osrm.keys()

Route information is stored under the `routes` key, which is a list of route objects.
Since we requested a single route, we access the first element and examine its structure.


In [ ]:
route_osrm = data_osrm["routes"][0]

route_osrm.keys()

Pull the key route properties out of the response.


In [ ]:
distance_osrm = route_osrm["distance"]
duration_osrm = route_osrm["duration"]
geometry_osrm = route_osrm["geometry"]

print(f"Distance (metres): {distance_osrm}")
print(f"Duration (seconds): {duration_osrm}")
print(f"Geometry: {geometry_osrm[:50]}...")

The route geometry is returned in encoded polyline format to reduce the size of the response.
To use it further (e.g. for visualisation), it must be decoded into a list of coordinates.


#### 2.1.6. Decoding the Geometry

Decode the route geometry from the encoded polyline format into a list of coordinates with the `polyline` library.


In [ ]:
decoded_route_osrm = polyline.decode(geometry_osrm)

The result is a list of coordinate pairs describing the route.
Since there may be many points, let's print just the first few:


In [ ]:
decoded_route_osrm[:5]

Each point is returned as a (latitude, longitude) pair, since `polyline.decode()` uses that order.

However, `shapely` expects coordinates in (longitude, latitude) order, so we need to swap them before creating the geometry.


In [ ]:
route_line_osrm = LineString([(lon, lat) for lat, lon in decoded_route_osrm])

#### 2.1.7. Creating a GeoDataFrame

Wrap the line in a `GeoDataFrame` so it can go on a map.


In [ ]:
route_gdf_osrm = gpd.GeoDataFrame(
    {"name": ["Route"]},
    geometry=[route_line_osrm],
    crs="EPSG:4326"
)

The same for the start and end points:


In [ ]:
points_gdf_osrm = gpd.GeoDataFrame(
    {"name": ["Start", "End"]},
    geometry=gpd.points_from_xy(
        [start_coords_osrm[0], end_coords_osrm[0]], 
        [start_coords_osrm[1], end_coords_osrm[1]]
    ),
    crs="EPSG:4326"
)

#### 2.1.8. Visualising the Result

And the route with its endpoints on an interactive map.


In [ ]:
m = route_gdf_osrm.explore(
    tiles="cartodbpositron",
    color="#FCDD9D",
    style_kwds={"weight": 5},
    tooltip="name"
)

points_gdf_osrm.explore(
    m=m,
    color="#A3B565",
    marker_kwds={"radius": 6},
    tooltip="name"
)

In this example, we covered the basic OSRM workflow – retrieving a route between two points. The service also supports additional capabilities such as distance matrices; these are described in the documentation.


### 2.2. OpenRouteService

**OpenRouteService (ORS)** is a routing and spatial analysis service built on OpenStreetMap data. Everything above carries over – build a request, check the status, parse the JSON – so the walkthrough is not repeated here. What follows is only what ORS does *differently*, and the one thing it does that OSRM cannot.

Three differences are worth knowing:

- the API key travels in a **request header**, not in the URL;
- the heavier endpoints want **POST** with a JSON body rather than GET with query parameters;
- routes come back as **GeoJSON**, so `gpd.GeoDataFrame.from_features()` reads them directly – no polyline to decode.

[Documentation](https://openrouteservice.org/dev/#/api-docs)

> **Note:** the cells below need a personal API key, so they do not run here. Register at [OpenRouteService](https://openrouteservice.org), copy the key from your dashboard, and run them locally. The placeholder is `your_ors_key`.
>
> Since you cannot see the output, these are the figures a working key returns, so you can check your own run against them:
>
> - the **route** from the Riesenrad to the Belvedere: about **4,160 metres**, roughly **10 minutes** by car;
> - the **distance matrix** over the four locations: a 4×4 table, with the Schönbrunn pair the furthest apart at 8–10 km;
> - the **isochrones**: three polygons, with `value` fields of 300, 600 and 900.
>
> If your results differ by more than a little, the likely culprit is the coordinate order – ORS expects longitude first.

#### 2.2.1. A Route

The key goes in the headers; the rest is the request you already know. Because the response is GeoJSON, the route reaches a `GeoDataFrame` in one call.

In [ ]:
ors_api_key = "your_ors_key"

url_directions_ors = "https://api.openrouteservice.org/v2/directions/driving-car"

# Coordinates (longitude, latitude)
start_coords_ors = [16.3956, 48.2166]    # Wiener Riesenrad
end_coords_ors = [16.3806, 48.1917]      # Belvedere

params_directions_ors = {
    "start": f"{start_coords_ors[0]},{start_coords_ors[1]}",
    "end": f"{end_coords_ors[0]},{end_coords_ors[1]}",
}
headers_directions_ors = {"Authorization": ors_api_key}

response_directions_ors = requests.get(
    url_directions_ors, params=params_directions_ors, headers=headers_directions_ors
)
response_directions_ors.status_code

In [ ]:
data_directions_ors = response_directions_ors.json()

# distance and duration live under properties -> summary
summary_ors = data_directions_ors["features"][0]["properties"]["summary"]
print(f"Distance (metres): {summary_ors['distance']}")
print(f"Duration (seconds): {summary_ors['duration']}")

# GeoJSON in, GeoDataFrame out - no polyline decoding needed
route_gdf_ors = gpd.GeoDataFrame.from_features(
    data_directions_ors["features"], crs="EPSG:4326"
)
route_gdf_ors.explore(tiles="cartodbpositron", color="#FCDD9D",
                      style_kwds={"weight": 5})

#### 2.2.2. Isochrones

This is the endpoint worth coming to ORS for: **isochrones**, the zones reachable from a point within a given travel time. The [previous section](networkAnalysis_2.ipynb) built the equivalent by hand from a local graph; here one request does it.

It is also the first **POST** we send. The parameters no longer fit in a URL, so they travel as a JSON body, and the headers gain a `Content-Type`. `range` is in seconds – 300, 600 and 900 are the 5, 10 and 15 minutes the course project uses.

In [ ]:
url_isochrones_ors = "https://api.openrouteservice.org/v2/isochrones/foot-walking"

start_coords_isochrones_ors = [16.3700, 48.2005]   # Karlsplatz

params_isochrones_ors = {
    "locations": [start_coords_isochrones_ors],
    "range": [300, 600, 900],          # 5, 10 and 15 minutes, in seconds
}
headers_isochrones_ors = {
    "Authorization": ors_api_key,
    "Content-Type": "application/json",
}

response_isochrones_ors = requests.post(
    url_isochrones_ors, json=params_isochrones_ors, headers=headers_isochrones_ors
)
response_isochrones_ors.status_code

The response is GeoJSON again: one feature per zone, each carrying the time it was drawn for in `value`.

In [ ]:
data_isochrones_ors = response_isochrones_ors.json()

isochrones_gdf_ors = gpd.GeoDataFrame.from_features(
    data_isochrones_ors["features"], crs="EPSG:4326"
)
print(isochrones_gdf_ors[["value"]])

isochrones_gdf_ors.explore(tiles="cartodbpositron")

#### 2.2.3. Distance Matrix

The same POST pattern returns distances between many points at once – the service-side equivalent of the matrix we assembled with Dijkstra in the [previous section](networkAnalysis_2.ipynb). `metrics` chooses what comes back: `distance`, `duration`, or both.

In [ ]:
url_matrix_ors = "https://api.openrouteservice.org/v2/matrix/driving-car"

locations_coords_ors = [
    [16.3725, 48.2082],   # Stephansplatz
    [16.3122, 48.1847],   # Schönbrunn Palace
    [16.3956, 48.2166],   # Wiener Riesenrad
    [16.3806, 48.1917],   # Belvedere
]

response_matrix_ors = requests.post(
    url_matrix_ors,
    json={"locations": locations_coords_ors, "metrics": ["distance"]},
    headers={"Authorization": ors_api_key, "Content-Type": "application/json"},
)

# a nested list: one row and one column per input location
print(response_matrix_ors.json()["distances"])

### 2.3. GraphHopper

**GraphHopper** is a third routing service on OpenStreetMap data, and it is here mainly to make a point: by now there is nothing new to learn. It takes a **GET** request like OSRM, wants its key as a **query parameter** like OSRM, and returns its geometry as an **encoded polyline** like OSRM. The whole of section 2.1 applies unchanged.

One thing does differ, and it is the classic way to lose an afternoon: **GraphHopper wants coordinates as `latitude,longitude`**, the reverse of OSRM and ORS. A route that comes back a few thousand kilometres away, or fails outright, is nearly always this.

[Documentation](https://docs.graphhopper.com/) · the key placeholder below is `your_graph_key`.

#### 2.3.1. The Three Services Side by Side

| | OSRM | OpenRouteService | GraphHopper |
| --- | --- | --- | --- |
| API key | not needed | header | query parameter |
| Coordinate order | lon, lat | lon, lat | **lat, lon** |
| Route geometry | encoded polyline | GeoJSON | encoded polyline |
| Request method | GET | GET and POST | GET |
| Distance matrix | yes | yes | yes |
| Isochrones | **no** | yes | yes |

Pick on capability and licence, not on syntax: the syntax is a morning's work in any of them, while the absence of an isochrone endpoint is not something you can code around.

The full request, parse and map in one cell – the same seven steps as section 2.1, with the coordinate order flipped where GraphHopper expects it:

In [ ]:
graphhopper_api_key = "your_graph_key"

url_directions_graph = "https://graphhopper.com/api/1/route"

# Coordinates (longitude, latitude), as everywhere else in this course
start_coords_graph = [16.3755, 48.1855]  # Wien Hauptbahnhof
end_coords_graph = [16.3725, 48.2082]    # Stephansplatz

# ... but GraphHopper wants them the other way round, as "lat,lon" strings
params_directions_graph = {
    "point": [
        f"{start_coords_graph[1]},{start_coords_graph[0]}",
        f"{end_coords_graph[1]},{end_coords_graph[0]}",
    ],
    "profile": "car",
    "key": graphhopper_api_key,
}

response_directions_graph = requests.get(url_directions_graph,
                                         params=params_directions_graph)
print(response_directions_graph.status_code)

route_graph = response_directions_graph.json()["paths"][0]
print(f"Distance (metres): {route_graph['distance']}")
print(f"Duration (seconds): {route_graph['time'] / 1000}")

# polyline again, and polyline.decode returns (lat, lon) - so swap for shapely
route_line_graph = LineString(
    [(lon, lat) for lat, lon in polyline.decode(route_graph["points"])]
)
route_gdf_graph = gpd.GeoDataFrame({"name": ["Route"]},
                                   geometry=[route_line_graph], crs="EPSG:4326")

route_gdf_graph.explore(tiles="cartodbpositron", color="#FCDD9D",
                        style_kwds={"weight": 5}, tooltip="name")

## Summary


In this section, we covered the fundamentals of **network analysis using external APIs**.

We learned:

- what an API is and how HTTP requests to web services are structured;
- how to read documentation and identify the required request parameters;
- how to send requests from Python using the `requests` library;
- how to parse responses and convert API results into `GeoDataFrame` objects for further analysis and visualisation.

The pattern is worth stating once more, because it was the same every time: **build the request, check the status, parse the JSON, turn it into geometry.** Three different services, and only the details changed – where the key goes, which method to use, whether the geometry arrives as GeoJSON or as an encoded polyline, and which way round the coordinates go.

External APIs are a convenient alternative to building and maintaining a local graph, as the computations are handled server-side on demand. What they are not is interchangeable: rate limits, licences and the set of available endpoints differ, and the isochrone row of the table above is the kind of difference that decides which service a project can use at all.